In [49]:
from data_engineering.flights.flight_type import get_flights, get_flights_from_iceberg
from datetime import date,timedelta
import polars as pl

start_date = date(2025,7,1)
end_date = date(2025,8,1)
df_flights = pl.DataFrame()
for target_date in [start_date + timedelta(days=i) for i in range((end_date - start_date).days)]:
    df_flights = pl.concat([df_flights, get_flights(target_date)], how="vertical")
df_flights

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str
"""010058""","""SUBNP""","""SU-BNP""",2025-07-01 15:14:19.210,"""HEAL""",2025-07-01 15:31:18.240,"""HEOC""",false,false,false,false,"""GLF4""",null,"""GULFSTREAM 4""","""A2"""
"""0100a2""","""MSR551""","""SU-GCE""",2025-07-01 13:09:30.800,"""HE30""",2025-07-01 13:28:17.190,"""HECA""",false,false,false,false,"""A332""",null,"""AIRBUS A-330-200""","""A5"""
"""0100a2""","""MSR542""","""SU-GCE""",2025-07-01 18:23:43.610,"""HE42""",2025-07-01 20:25:19.410,"""LTFM""",false,false,false,false,"""A332""",null,"""AIRBUS A-330-200""","""A5"""
"""0100a2""","""MSR543""","""SU-GCE""",2025-07-01 21:48:55.030,"""LTFM""",2025-07-01 23:21:37.740,"""HEAZ""",false,false,false,false,"""A332""",null,"""AIRBUS A-330-200""","""A5"""
"""0100a3""","""MSR552""","""SU-GCF""",2025-07-01 07:38:19.150,"""HECA""",2025-07-01 07:49:18.290,"""HE30""",false,false,false,false,"""A332""",null,"""AIRBUS A-330-200""","""A5"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""e94c8a""","""BOV795""",null,2025-07-31 18:42:42.730,"""AR-0583""",2025-07-31 19:13:21.180,"""PY-0033""",null,null,null,null,null,null,null,"""A2"""
"""e94c8e""","""BOV767""","""CP-3214""",2025-07-31 02:25:36.910,"""KMIA""",2025-07-31 03:57:41.260,"""JM-0016""",false,false,false,false,"""A332""",null,"""AIRBUS A-330-200""","""A5"""
"""e94c8f""","""BOV736""","""CP-3215""",2025-07-31 10:44:41.510,"""SLLJ""",2025-07-31 15:39:38.710,"""SBGR""",false,false,false,false,"""B738""",null,"""BOEING 737-800""","""A3"""


In [50]:
df_flights = df_flights.filter((pl.col("ladd") == True) | (pl.col("pia") == True))

In [51]:
df_flights = df_flights.with_columns(
    pl.concat_str(
        [
            pl.col("icao"),
            pl.col("takeoff_time").dt.strftime("%Y-%m-%d %H:%M:%S"),
        ],
        separator="_",
    ).alias("flight_id")
)

In [52]:
df_flights = df_flights.select("flight_id", "icao", "takeoff_time", "landing_time", "landing_airport_ident", "takeoff_airport_ident", "ladd", "pia", "aircraft_type")

In [59]:
next_ids = (
    df_flights.join(
        df_flights.select(
            pl.col("flight_id").alias("next_flight_id"),
            pl.col("icao").alias("next_icao"),
            pl.col("pia").alias("next_pia"),
            pl.col("aircraft_type").alias("next_aircraft_type"),
            "takeoff_airport_ident",
            pl.col("takeoff_time").alias("next_takeoff_time"),
        ),
        left_on="landing_airport_ident",
        right_on="takeoff_airport_ident",
        how="left",
    )
    .filter(
        (pl.col("next_takeoff_time") > pl.col("landing_time"))
        & (
            (pl.col("next_icao") == pl.col("icao"))
            | (
                pl.col("next_pia")
                & pl.col("aircraft_type").is_not_null()
                & pl.col("next_aircraft_type").is_not_null()
                & (pl.col("aircraft_type") != "")
                & (pl.col("next_aircraft_type") != "")
                & (pl.col("next_aircraft_type") == pl.col("aircraft_type"))
            )
        )
    )
    .group_by("flight_id")
    .agg(
        pl.col("next_flight_id")
        .unique()
        .alias("next_flight_ids")
    )
)

df = (
    df_flights
    .join(next_ids, on="flight_id", how="left")
    .with_columns(
        pl.col("next_flight_ids")
        .fill_null(pl.lit([], dtype=pl.List(pl.String)))
    )
)

In [60]:
df

flight_id,icao,takeoff_time,landing_time,landing_airport_ident,takeoff_airport_ident,ladd,pia,aircraft_type,next_flight_ids
str,str,datetime[ms],datetime[ms],str,str,bool,bool,str,list[str]
"""020141_2025-07-01 19:01:44""","""020141""",2025-07-01 19:01:44.180,2025-07-01 19:30:50.550,"""MA-0008""","""GMME""",true,false,"""C56X""","[""020141_2025-07-07 10:22:33""]"
"""020141_2025-07-01 19:56:59""","""020141""",2025-07-01 19:56:59.980,2025-07-01 20:41:36.320,"""GMFF""","""MA-0009""",true,false,"""C56X""","[""020141_2025-07-01 20:53:49"", ""020141_2025-07-10 10:48:39"", ""020141_2025-07-11 20:20:37""]"
"""020141_2025-07-01 20:53:49""","""020141""",2025-07-01 20:53:49.290,2025-07-01 21:14:54.570,"""GMME""","""GMFF""",true,false,"""C56X""","[""020141_2025-07-03 08:46:06"", ""020141_2025-07-07 08:37:34"", … ""020141_2025-07-16 19:31:06""]"
"""04c293_2025-07-01 13:54:03""","""04c293""",2025-07-01 13:54:03.490,2025-07-01 14:22:37.320,"""OMDW""","""OOII""",true,false,"""C680""","[""04c293_2025-07-01 15:54:44"", ""04c293_2025-07-21 23:34:34""]"
"""04c293_2025-07-01 15:54:44""","""04c293""",2025-07-01 15:54:44.770,2025-07-01 16:25:40.990,"""OOFD""","""OMDW""",true,false,"""C680""",[]
…,…,…,…,…,…,…,…,…,…
"""e49f89_2025-07-31 22:32:13""","""e49f89""",2025-07-31 22:32:13.920,2025-07-31 22:46:32.980,"""BR-0414""","""SBMT""",true,false,"""BE30""",[]
"""e49f89_2025-07-31 23:28:51""","""e49f89""",2025-07-31 23:28:51.030,2025-07-31 23:50:44.040,"""SSNU""","""SBJD""",true,false,"""BE30""",[]
"""e4a0d5_2025-07-31 12:08:05""","""e4a0d5""",2025-07-31 12:08:05.550,2025-07-31 12:59:48.920,"""BR-0513""","""SDLY""",true,false,"""E35L""",[]


In [55]:
import networkx as nx

G = nx.DiGraph()

# Add all flights as nodes
for row in df.iter_rows(named=True):
    G.add_node(
        row["flight_id"],
        icao=row["icao"],
        takeoff_time=row["takeoff_time"],
        landing_time=row["landing_time"],
        takeoff_airport_ident=row["takeoff_airport_ident"],
        landing_airport_ident=row["landing_airport_ident"],
        pia=row["pia"],
    )

# Add edges to possible next flights
for row in df.select(["flight_id", "next_flight_ids"]).iter_rows(named=True):
    src = row["flight_id"]

    for dst in row["next_flight_ids"]:
        G.add_edge(src, dst)

In [57]:
import networkx as nx

# Keep only nodes that are PIA flights
pia_nodes = {
    row["flight_id"]
    for row in df.filter(pl.col("pia")).select("flight_id").iter_rows(named=True)
}

# Find every node connected to a PIA node
keep_nodes = set()

for pia_node in pia_nodes:
    keep_nodes.update(nx.descendants(G, pia_node))
    keep_nodes.update(nx.ancestors(G, pia_node))
    keep_nodes.add(pia_node)

# Subgraph containing only paths touching >=1 PIA flight
G = G.subgraph(keep_nodes).copy()

In [61]:
df_flights.filter(pl.col("pia") == True).get_column("icao").unique()

icao
str
"""a0399f"""
"""a10410"""
"""a4f0af"""
"""a03ba0"""
"""a03bff"""
…
"""a1c15a"""
"""a03c82"""
"""a4f4f6"""


In [62]:
subgraphs = [
    G.subgraph(c).copy()
    for c in nx.weakly_connected_components(G)
]
len(subgraphs)

48

In [63]:
target = "a10151_2025-07-31 16:21:28"

depth = 2

nodes = set()

for node, dist in nx.single_source_shortest_path_length(
    G,
    target,
    cutoff=depth,
).items():
    nodes.add(node)

for node, dist in nx.single_source_shortest_path_length(
    G.reverse(),
    target,
    cutoff=depth,
).items():
    nodes.add(node)

G_small = G.subgraph(nodes).copy()

In [64]:
import networkx as nx
import plotly.graph_objects as go

target = "a10151_2025-07-31 16:21:28"

# Small neighborhood around target
nodes = set()

nodes.update(
    nx.single_source_shortest_path_length(
        G,
        target,
        cutoff=2,
    ).keys()
)

nodes.update(
    nx.single_source_shortest_path_length(
        G.reverse(),
        target,
        cutoff=2,
    ).keys()
)

G_small = G.subgraph(nodes).copy()

# Layout
pos = nx.spring_layout(G_small, seed=1)

# Edges
edge_x = []
edge_y = []

for u, v in G_small.edges():
    x0, y0 = pos[u]
    x1, y1 = pos[v]

    edge_x += [x0, x1, None]
    edge_y += [y0, y1, None]

edge_trace = go.Scatter(
    x=edge_x,
    y=edge_y,
    line=dict(width=1),
    hoverinfo="none",
    mode="lines",
)

# Nodes
node_x = []
node_y = []
node_text = []
node_color = []

for node in G_small.nodes():
    x, y = pos[node]

    node_x.append(x)
    node_y.append(y)
    node_text.append(node)

    if node == target:
        node_color.append("red")
    elif G_small.nodes[node].get("pia"):
        node_color.append("orange")
    else:
        node_color.append("lightblue")

node_trace = go.Scatter(
    x=node_x,
    y=node_y,
    text=node_text,
    mode="markers+text",
    hoverinfo="text",
    marker=dict(
        size=10,
        color=node_color,
    ),
)

fig = go.Figure(
    data=[edge_trace, node_trace],
)

fig.update_layout(
    showlegend=False,
    hovermode="closest",
)

fig.show()

In [66]:
g_0 = subgraphs[0]
g_0

In [70]:
target = "a10151_2025-07-31 16:21:28"

ancestors = list(nx.ancestors(G, target))
descendants = list(nx.descendants(G, target))

paths = []

for src in ancestors + [target]:
    for dst in descendants + [target]:
        if src != dst:
            try:
                path = nx.shortest_path(G, src, dst)

                if target in path:
                    paths.append(path)

                    if len(paths) >= 2:
                        break

            except nx.NetworkXNoPath:
                pass

    if len(paths) >= 2:
        break

print(paths)

[['c05d79_2025-07-18 16:35:47', 'c02df1_2025-07-19 10:34:52', 'c02df1_2025-07-20 14:14:22', 'ab6449_2025-07-20 15:29:30', 'a7b13f_2025-07-20 21:57:15', 'a7b13f_2025-07-21 01:10:18', 'a7b13f_2025-07-21 13:02:40', 'a98739_2025-07-21 14:32:53', 'a03bc7_2025-07-24 03:24:51', 'a10151_2025-07-26 19:31:31', 'a10151_2025-07-31 16:21:28'], ['ac42a3_2025-07-13 16:51:38', 'a4f4f6_2025-07-14 15:43:58', 'a10151_2025-07-16 20:55:11', 'a10151_2025-07-31 16:21:28']]


In [69]:
from data_engineering.flights.flight_type import Flight, flights_df_to_flights
flights_objs:dict[str, Flight] = dict()
for flight in df_flights.iter_rows(named=True):
     flights_objs[flight["flight_id"]] = Flight(**flight)

In [73]:
df_path_0 = pl.DataFrame()
for flight_id in paths[0]:
    flight = flights_objs[flight_id]
    df_path_0 = pl.concat(
        [
            df_path_0,
            pl.DataFrame(
                {
                    "flight_id": flight.flight_id,
                    "icao": flight.icao,
                    "takeoff_time": flight.takeoff_time,
                    "landing_time": flight.landing_time,
                    "takeoff_airport_ident": flight.takeoff_airport_ident,
                    "landing_airport_ident": flight.landing_airport_ident,
                    "ladd": flight.ladd,
                    "pia": flight.pia,
                    "aircraft_type": flight.aircraft_type,
                }
            ),
        ],
        how="vertical",
    )
df_path_0

flight_id,icao,takeoff_time,landing_time,takeoff_airport_ident,landing_airport_ident,ladd,pia,aircraft_type
str,str,datetime[μs],datetime[μs],str,str,bool,bool,str
"""c05d79_2025-07-18 16:35:47""","""c05d79""",2025-07-18 16:35:47.050,2025-07-18 17:34:37.970,"""CYOW""","""CYSJ""",true,false,"""F2TH"""
"""c02df1_2025-07-19 10:34:52""","""c02df1""",2025-07-19 10:34:52.650,2025-07-19 10:50:37.180,"""CYSJ""","""CYQM""",true,false,"""F2TH"""
"""c02df1_2025-07-20 14:14:22""","""c02df1""",2025-07-20 14:14:22.760,2025-07-20 15:24:31.380,"""CYQM""","""KBOS""",true,false,"""F2TH"""
"""ab6449_2025-07-20 15:29:30""","""ab6449""",2025-07-20 15:29:30.540,2025-07-20 20:26:37.380,"""KBOS""","""KBZN""",true,false,"""F2TH"""
"""a7b13f_2025-07-20 21:57:15""","""a7b13f""",2025-07-20 21:57:15.700,2025-07-21 00:40:14.690,"""KBZN""","""KBNA""",true,false,"""F2TH"""
…,…,…,…,…,…,…,…,…
"""a7b13f_2025-07-21 13:02:40""","""a7b13f""",2025-07-21 13:02:40.930,2025-07-21 14:28:32.500,"""KTYS""","""KTEB""",true,false,"""F2TH"""
"""a98739_2025-07-21 14:32:53""","""a98739""",2025-07-21 14:32:53.270,2025-07-21 18:50:15.390,"""KTEB""","""KJAC""",true,false,"""F2TH"""
"""a03bc7_2025-07-24 03:24:51""","""a03bc7""",2025-07-24 03:24:51.020,2025-07-24 05:23:36.830,"""KJAC""","""KCMA""",false,true,null


In [74]:
paths[0]

['c05d79_2025-07-18 16:35:47',
 'c02df1_2025-07-19 10:34:52',
 'c02df1_2025-07-20 14:14:22',
 'ab6449_2025-07-20 15:29:30',
 'a7b13f_2025-07-20 21:57:15',
 'a7b13f_2025-07-21 01:10:18',
 'a7b13f_2025-07-21 13:02:40',
 'a98739_2025-07-21 14:32:53',
 'a03bc7_2025-07-24 03:24:51',
 'a10151_2025-07-26 19:31:31',
 'a10151_2025-07-31 16:21:28']

In [81]:
a = set(df_flights.filter(pl.col("pia") == True).get_column("icao").unique())
"a03bc7" in a

True

In [ ]:
# I should make dict with key being flight_id and value being Flight Object 

In [ ]:
# how do I deal with getting back to the icaos. 
# I can iterate for through each flight
# I'm going to try with assuuming flight data is perfect first just to see how far I get. 